In [2]:
#Inference_using_3_Model_3
#!/usr/bin/env python
# coding: utf-8

"""
Enhanced Medical Image Analysis Pipeline for Anemia Detection - RGB/HSV/LAB CSV Output
=====================================================================================
This script processes medical images to detect eye parts, color patches, and perform
segmentation analysis. It outputs THREE CSVs: RGB values, HSV values, and CIE L*a*b*
values for the exact same patch bounding boxes and ordering.

Color space notes:
- HSV (OpenCV): H∈[0,179], S∈[0,255], V∈[0,255]
- LAB (OpenCV 8-bit input): L∈[0,255] (scaled from 0..100), a∈[0,255] (128≈0), b∈[0,255] (128≈0)

Author: Medical AI Team
Date: 2025
"""

import os
import sys
import cv2
import torch
import numpy as np
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Tuple, Optional, Dict, Any, Union
from dataclasses import dataclass

from ultralytics import YOLO
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import matplotlib.pyplot as plt


@dataclass
class Config:
    """Configuration class for the medical image analysis pipeline."""
    
    # Input/Output Paths
    input_folder: Path = Path(r"/home/khushi/Pixonate/New_Anemia/New/all_Sort_data/Full_data")
    output_overlay_dir: Path = Path(r"/home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/overlay_output")
    output_csv_file: str = "/home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/RGB.csv"             # RGB CSV
    output_csv_file_hsv: str = "/home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/HSV.csv"     # HSV CSV
    output_csv_file_lab: str = "/home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/LAB.csv"     # LAB CSV (NEW)
    log_file: str = "/home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/medical_analysis.log"
    
    # Model Paths
    yolo_weights: str = r"/home/khushi/Pixonate/New_Anemia/Models/Finetuned model/Eye model/Fbest.pt"
    color_weights: str = r"/home/khushi/Pixonate/New_Anemia/Models/Yolo model/Color boxes model/best1.pt"
    seg_weights: str = r"/home/khushi/Pixonate/New_Anemia/Models/Finetuned model/Unet Eye model/Nbest_model.pth"
    
    # Model Parameters
    conf_thresh: float = 0.7
    yolo_imgsz: int = 1024
    net_size: int = 256
    alpha: float = 0.5
    chart_rows: int = 4
    chart_cols: int = 6
    
    # Processing Options
    use_median: bool = True  
    save_overlays: bool = True
    show_plots: bool = True
    
    # Device
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Logger:
    """Enhanced logger for the medical analysis pipeline."""
    
    def __init__(self, log_file: str = "medical_analysis.log", level=logging.INFO):
        self.logger = logging.getLogger("MedicalAnalysis")
        self.logger.setLevel(level)
        
        # Clear existing handlers
        for handler in self.logger.handlers[:]:
            self.logger.removeHandler(handler)
        
        # Create formatters
        detailed_formatter = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - [%(funcName)s:%(lineno)d] - %(message)s'
        )
        console_formatter = logging.Formatter(
            '%(asctime)s - %(levelname)s - %(message)s'
        )
        
        # File handler with detailed logging
        file_handler = logging.FileHandler(log_file, mode='w', encoding='utf-8')
        file_handler.setLevel(logging.DEBUG)
        file_handler.setFormatter(detailed_formatter)
        
        # Console handler with cleaner output
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        console_handler.setFormatter(console_formatter)
        
        self.logger.addHandler(file_handler)
        self.logger.addHandler(console_handler)
    
    def info(self, message: str):
        self.logger.info(message)
    
    def debug(self, message: str):
        self.logger.debug(message)
    
    def warning(self, message: str):
        self.logger.warning(message)
    
    def error(self, message: str):
        self.logger.error(message)
    
    def critical(self, message: str):
        self.logger.critical(message)


@dataclass
class ColorPatch:
    """Data class for color patch information."""
    uid: int
    rgb_values: Tuple[int, int, int]
    bbox: Tuple[int, int, int, int]  # x1, y1, x2, y2
    center: Tuple[int, int]  # cx, cy
    area: int


@dataclass
class EyeSegmentation:
    """Data class for eye segmentation results."""
    rgb_values: Tuple[int, int, int]
    contour_area: float
    mask_pixel_count: int


class CSVResultManager:
    """Manages CSV output and results processing (RGB)."""
    
    def __init__(self, config: Config, logger: Logger):
        self.config = config
        self.logger = logger
        self.csv_file = None
        self.csv_writer = None
        self.csv_headers_written = False
    
    def initialize_csv(self) -> bool:
        """Initialize CSV file with headers."""
        try:
            import importlib
            csv_module = importlib.import_module('csv')
            
            self.csv_file = open(self.config.output_csv_file, 'w', newline='', encoding='utf-8')
            self.csv_writer = csv_module.writer(self.csv_file)
            
            headers = ['image_name']
            for i in range(1, 25):  # UIDs 1-24
                headers.extend([f'patch_{i}_r', f'patch_{i}_g', f'patch_{i}_b'])
            headers.extend(['total_features', 'avg_patch_r', 'avg_patch_g', 'avg_patch_b'])
            
            self.csv_writer.writerow(headers)
            self.csv_headers_written = True
            
            self.logger.info(f"RGB CSV initialized with {len(headers)} columns: {self.config.output_csv_file}")
            return True
            
        except Exception as e:
            self.logger.error(f"Failed to initialize RGB CSV: {str(e)}")
            return False
    
    def write_analysis_result(self, image_name: str, color_patches: List[ColorPatch],
                              processing_success: bool) -> None:
        """Write a single analysis result to RGB CSV."""
        try:
            if not self.csv_headers_written:
                self.logger.error("RGB CSV not initialized")
                return
            
            row_data = [image_name]
            
            patch_dict = {patch.uid: patch for patch in color_patches}
            patch_rgbs = []
            
            for uid in range(1, 25):  # UIDs 1-24
                if uid in patch_dict:
                    patch = patch_dict[uid]
                    row_data.extend([patch.rgb_values[0], patch.rgb_values[1], patch.rgb_values[2]])
                    patch_rgbs.extend(patch.rgb_values)
                else:
                    row_data.extend([0, 0, 0])  # Missing patch
            
            total_features = len(color_patches)
            if patch_rgbs:
                r_values = [patch_rgbs[i] for i in range(0, len(patch_rgbs), 3)]
                g_values = [patch_rgbs[i] for i in range(1, len(patch_rgbs), 3)]
                b_values = [patch_rgbs[i] for i in range(2, len(patch_rgbs), 3)]
                avg_r = sum(r_values) / len(r_values) if r_values else 0
                avg_g = sum(g_values) / len(g_values) if g_values else 0
                avg_b = sum(b_values) / len(b_values) if b_values else 0
            else:
                avg_r = avg_g = avg_b = 0
            
            row_data.extend([total_features, avg_r, avg_g, avg_b])
            
            # Expected columns = 1 + (24*3) + 4 = 77
            if len(row_data) != 77:
                self.logger.warning(f"RGB row data length mismatch: {len(row_data)} (expected 77)")
            
            self.csv_writer.writerow(row_data)
            self.csv_file.flush()
            
            self.logger.debug(f"RGB CSV row written for {image_name}")
            
        except Exception as e:
            self.logger.error(f"Failed to write RGB CSV row for {image_name}: {str(e)}")
    
    def finalize_csv(self) -> None:
        """Close CSV file and finalize output."""
        try:
            if self.csv_file:
                self.csv_file.close()
                self.logger.info(f"RGB CSV file finalized: {self.config.output_csv_file}")
        except Exception as e:
            self.logger.error(f"Error finalizing RGB CSV: {str(e)}")


class HSVCSVResultManager:
    """Writes HSV values (same patch UIDs/bboxes) to a separate CSV."""
    def __init__(self, config: Config, logger: Logger):
        self.config = config
        self.logger = logger
        self.csv_file = None
        self.csv_writer = None
        self.csv_headers_written = False

    def initialize_csv(self) -> bool:
        try:
            import importlib
            csv_module = importlib.import_module('csv')
            self.csv_file = open(self.config.output_csv_file_hsv, 'w', newline='', encoding='utf-8')
            self.csv_writer = csv_module.writer(self.csv_file)

            headers = ['image_name']
            for i in range(1, 25):  # UIDs 1..24
                headers.extend([f'patch_{i}_h', f'patch_{i}_s', f'patch_{i}_v'])
            headers.extend(['total_features', 'avg_patch_h', 'avg_patch_s', 'avg_patch_v'])

            self.csv_writer.writerow(headers)
            self.csv_headers_written = True
            self.logger.info(f"HSV CSV initialized with {len(headers)} columns: {self.config.output_csv_file_hsv}")
            return True
        except Exception as e:
            self.logger.error(f"Failed to initialize HSV CSV: {str(e)}")
            return False

    def write_hsv_result(self, image_name: str, hsv_by_uid: Dict[int, Tuple[int,int,int]],
                         total_features: int) -> None:
        try:
            if not self.csv_headers_written:
                self.logger.error("HSV CSV not initialized")
                return

            row = [image_name]

            # 24 patches × (H,S,V)
            h_vals, s_vals, v_vals = [], [], []
            for uid in range(1, 25):
                if uid in hsv_by_uid:
                    h, s, v = hsv_by_uid[uid]
                    row.extend([h, s, v])
                    h_vals.append(h); s_vals.append(s); v_vals.append(v)
                else:
                    row.extend([0, 0, 0])

            # summary
            if h_vals:
                avg_h = sum(h_vals) / len(h_vals)
                avg_s = sum(s_vals) / len(s_vals)
                avg_v = sum(v_vals) / len(v_vals)
            else:
                avg_h = avg_s = avg_v = 0

            row.extend([total_features, avg_h, avg_s, avg_v])

            self.csv_writer.writerow(row)
            self.csv_file.flush()
            self.logger.debug(f"HSV CSV row written for {image_name}")
        except Exception as e:
            self.logger.error(f"Failed to write HSV CSV row for {image_name}: {str(e)}")

    def finalize_csv(self) -> None:
        try:
            if self.csv_file:
                self.csv_file.close()
                self.logger.info(f"HSV CSV file finalized: {self.config.output_csv_file_hsv}")
        except Exception as e:
            self.logger.error(f"Error finalizing HSV CSV: {str(e)}")


class LABCSVResultManager:
    """Writes LAB values (same patch UIDs/bboxes) to a separate CSV."""
    def __init__(self, config: Config, logger: Logger):
        self.config = config
        self.logger = logger
        self.csv_file = None
        self.csv_writer = None
        self.csv_headers_written = False

    def initialize_csv(self) -> bool:
        try:
            import importlib
            csv_module = importlib.import_module('csv')
            self.csv_file = open(self.config.output_csv_file_lab, 'w', newline='', encoding='utf-8')
            self.csv_writer = csv_module.writer(self.csv_file)

            headers = ['image_name']
            for i in range(1, 25):  # UIDs 1..24
                headers.extend([f'patch_{i}_l', f'patch_{i}_a', f'patch_{i}_b'])
            headers.extend(['total_features', 'avg_patch_l', 'avg_patch_a', 'avg_patch_b'])

            self.csv_writer.writerow(headers)
            self.csv_headers_written = True
            self.logger.info(f"LAB CSV initialized with {len(headers)} columns: {self.config.output_csv_file_lab}")
            return True
        except Exception as e:
            self.logger.error(f"Failed to initialize LAB CSV: {str(e)}")
            return False

    def write_lab_result(self, image_name: str, lab_by_uid: Dict[int, Tuple[int,int,int]],
                         total_features: int) -> None:
        try:
            if not self.csv_headers_written:
                self.logger.error("LAB CSV not initialized")
                return

            row = [image_name]

            # 24 patches × (L,a,b)
            l_vals, a_vals, b_vals = [], [], []
            for uid in range(1, 25):
                if uid in lab_by_uid:
                    l, a, b = lab_by_uid[uid]
                    row.extend([l, a, b])
                    l_vals.append(l); a_vals.append(a); b_vals.append(b)
                else:
                    row.extend([0, 0, 0])

            # summary
            if l_vals:
                avg_l = sum(l_vals) / len(l_vals)
                avg_a = sum(a_vals) / len(a_vals)
                avg_b = sum(b_vals) / len(b_vals)
            else:
                avg_l = avg_a = avg_b = 0

            row.extend([total_features, avg_l, avg_a, avg_b])

            self.csv_writer.writerow(row)
            self.csv_file.flush()
            self.logger.debug(f"LAB CSV row written for {image_name}")
        except Exception as e:
            self.logger.error(f"Failed to write LAB CSV row for {image_name}: {str(e)}")

    def finalize_csv(self) -> None:
        try:
            if self.csv_file:
                self.csv_file.close()
                self.logger.info(f"LAB CSV file finalized: {self.config.output_csv_file_lab}")
        except Exception as e:
            self.logger.error(f"Error finalizing LAB CSV: {str(e)}")


class ModelManager:
    """Manages loading and validation of all models."""
    
    def __init__(self, config: Config, logger: Logger):
        self.config = config
        self.logger = logger    
        self.yolo_model = None
        self.color_model = None
        self.seg_model = None
        
    def validate_model_files(self) -> bool:
        """Validate that all model files exist."""
        model_files = [
            ("YOLO Eye Detection", self.config.yolo_weights),
            ("YOLO Color Detection", self.config.color_weights),
            ("Segmentation Model", self.config.seg_weights)
        ]
        
        all_valid = True
        for name, path in model_files:
            if not Path(path).exists():
                self.logger.error(f"{name} file not found: {path}")
                all_valid = False
            else:
                self.logger.debug(f"{name} file validated: {path}")
        
        return all_valid
    
    def load_models(self) -> bool:
        """Load all required models with error handling."""
        try:
            self.logger.info("=" * 50)
            self.logger.info("INITIALIZING MODEL LOADING")
            self.logger.info("=" * 50)
            
            # Validate files first
            if not self.validate_model_files():
                self.logger.critical("Model file validation failed")
                return False
            
            # Load YOLO models
            self.logger.info("Loading YOLO eye detection model...")
            self.yolo_model = YOLO(self.config.yolo_weights)
            self.logger.info("✓ YOLO eye detection model loaded successfully")
            
            self.logger.info("Loading YOLO color patch detection model...")
            self.color_model = YOLO(self.config.color_weights)
            self.logger.info("✓ YOLO color patch model loaded successfully")
            
            # Load segmentation model
            self.logger.info("Loading U-Net segmentation model...")
            self.seg_model = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1)
            state_dict = torch.load(self.config.seg_weights, map_location=self.config.device)
            self.seg_model.load_state_dict(state_dict)
            self.seg_model.to(self.config.device).eval()
            self.logger.info(f"✓ U-Net segmentation model loaded on {self.config.device}")
            
            self.logger.info("=" * 50)
            self.logger.info("ALL MODELS LOADED SUCCESSFULLY!")
            self.logger.info("=" * 50)
            return True
            
        except Exception as e:
            self.logger.critical(f"Critical error loading models: {str(e)}")
            return False


class ImageProcessor:
    """Handles all image processing operations."""
    
    def __init__(self, config: Config, logger: Logger, model_manager: ModelManager):
        self.config = config
        self.logger = logger
        self.models = model_manager
        self.seg_transform = self._create_segmentation_transform()
        
    def _create_segmentation_transform(self):
        """Create image preprocessing pipeline for segmentation."""
        self.logger.debug("Creating segmentation transform pipeline")
        return A.Compose([
            A.LongestMaxSize(max_size=self.config.net_size),
            A.PadIfNeeded(
                min_height=self.config.net_size,
                min_width=self.config.net_size,
                border_mode=cv2.BORDER_REFLECT
            ),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    
    def segment_crop(self, crop: np.ndarray) -> np.ndarray:
        """
        Perform segmentation on image crop.
        
        Args:
            crop: Input image crop as numpy array
            
        Returns:
            Binary mask of same dimensions as input
        """
        try:
            h, w = crop.shape[:2]
            self.logger.debug(f"Segmenting crop: {w}x{h} pixels")
            
            # Preprocess image
            prep = self.seg_transform(image=crop)
            inp = prep["image"].unsqueeze(0).to(self.config.device)
            
            # Run inference
            with torch.no_grad():
                prob = torch.sigmoid(self.models.seg_model(inp))[0, 0].cpu().numpy()
            
            # Undo padding and resize to original dimensions
            scale = self.config.net_size / max(h, w)
            nw, nh = int(w * scale), int(h * scale)
            pw, ph = self.config.net_size - nw, self.config.net_size - nh
            left, top = pw // 2, ph // 2
            
            prob_crop = prob[top:top + nh, left:left + nw]
            mask256 = (prob_crop > 0.5).astype(np.uint8)
            
            final_mask = cv2.resize(mask256, (w, h), interpolation=cv2.INTER_NEAREST)
            self.logger.debug(f"Segmentation completed - mask pixels: {np.sum(final_mask)}")
            
            return final_mask
            
        except Exception as e:
            self.logger.error(f"Segmentation failed: {str(e)}")
            return np.zeros((crop.shape[0], crop.shape[1]), dtype=np.uint8)
    
    def detect_objects(self, img_path: str) -> Tuple[Any, Any]:
        """
        Run YOLO detection on image.
        
        Args:
            img_path: Path to image file
            
        Returns:
            Tuple of (eye_results, color_results)
        """
        try:
            self.logger.debug(f"Running object detection on: {img_path}")
            
            # Eye part detection
            eye_results = self.models.yolo_model.predict(
                img_path,
                conf=self.config.conf_thresh,
                imgsz=self.config.yolo_imgsz,
                verbose=False
            )[0]
            
            # Color patch detection
            color_results = self.models.color_model.predict(
                img_path,
                conf=self.config.conf_thresh,
                imgsz=self.config.yolo_imgsz,
                verbose=False
            )[0]
            
            self.logger.debug(f"Detection complete - Eyes: {len(eye_results.boxes)}, Colors: {len(color_results.boxes)}")
            return eye_results, color_results
            
        except Exception as e:
            self.logger.error(f"Object detection failed: {str(e)}")
            return None, None


class ColorAnalyzer:
    """Handles color patch analysis and clustering."""
    
    def __init__(self, config: Config, logger: Logger):
        self.config = config
        self.logger = logger
    
    def cluster_patches_into_rows(self, centroids: List[Tuple]) -> List[List]:
        """
        Cluster color patches into rows using K-means.
        
        Args:
            centroids: List of (cx, cy, box) tuples
        
        Returns:
            List of rows containing grouped patches
        """
        try:
            num_patches = len(centroids)
            self.logger.debug(f"Clustering {num_patches} patches into rows")
            
            if num_patches < self.config.chart_rows:
                self.logger.warning(f"Insufficient patches ({num_patches}) for {self.config.chart_rows} rows")
                return [centroids]
            
            ys = np.array([[cy] for _, cy, _ in centroids], dtype=np.float32)
            criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
            _, labels, centers = cv2.kmeans(
                ys, self.config.chart_rows, None, criteria, 10, cv2.KMEANS_PP_CENTERS
            )
            
            order = np.argsort(centers.flatten())
            lbl2row = {orig: i for i, orig in enumerate(order)}
            
            rows = [[] for _ in range(self.config.chart_rows)]
            for (cx, cy, box), lbl in zip(centroids, labels.flatten()):
                rows[lbl2row[int(lbl)]].append((cx, cy, box))
            
            for i, row in enumerate(rows):
                self.logger.debug(f"Row {i+1}: {len(row)} patches")
            
            return rows
            
        except Exception as e:
            self.logger.error(f"Clustering failed: {str(e)}")
            return [centroids]
    
    def determine_patch_ordering(self, ordered_patches: List, img_rgb: np.ndarray) -> bool:
        """
        Determine ascending vs descending order based on bottom-left patch color.
        
        Args:
            ordered_patches: List of ordered patch detections
            img_rgb: Original RGB image
            
        Returns:
            True for ascending order, False for descending
        """
        try:
            total = min(self.config.chart_rows * self.config.chart_cols, len(ordered_patches))
            bl_idx = (self.config.chart_rows - 1) * self.config.chart_cols  # bottom-left index
            
            if len(ordered_patches) <= bl_idx:
                self.logger.debug("Using default ascending order")
                return True
            
            x1b, y1b, x2b, y2b = map(int, ordered_patches[bl_idx].xyxy[0].cpu().numpy())
            patch_bl = img_rgb[y1b:y2b, x1b:x2b]
            
            if self.config.use_median:
                mean_bl = tuple(int(v) for v in np.median(patch_bl, axis=(0, 1)))
            else:
                mean_bl = tuple(int(v) for v in patch_bl.mean(axis=(0, 1)))
            
            blue = np.array([0, 0, 255])
            white = np.array([255, 255, 255])
            dist_blue = np.linalg.norm(np.array(mean_bl) - blue)
            dist_white = np.linalg.norm(np.array(mean_bl) - white)
            
            ascending = (dist_white <= dist_blue)
            
            self.logger.debug(f"Bottom-left patch analysis:")
            self.logger.debug(f"  RGB: {mean_bl}")
            self.logger.debug(f"  Distance to blue: {dist_blue:.2f}")
            self.logger.debug(f"  Distance to white: {dist_white:.2f}")
            self.logger.debug(f"  Order: {'ascending' if ascending else 'descending'}")
            
            return ascending
            
        except Exception as e:
            self.logger.error(f"Order determination failed: {str(e)}")
            return True


class MedicalImageAnalyzer:
    """Main analyzer orchestrating the complete pipeline."""
    
    def __init__(self, config: Config):
        self.config = config
        self.logger = Logger(self.config.log_file)
        self.model_manager = ModelManager(self.config, self.logger)
        self.image_processor = ImageProcessor(self.config, self.logger, self.model_manager)
        self.color_analyzer = ColorAnalyzer(self.config, self.logger)
        self.csv_manager = CSVResultManager(self.config, self.logger)
        self.csv_manager_hsv = HSVCSVResultManager(self.config, self.logger)
        self.csv_manager_lab = LABCSVResultManager(self.config, self.logger)  # NEW
        
    def initialize_system(self) -> bool:
        """Initialize the complete analysis system."""
        try:
            self.logger.info("Medical Image Analysis Pipeline Starting")
            self.logger.info("=" * 70)
            self.logger.info(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            self.logger.info(f"Device: {self.config.device}")
            self.logger.info(f"Input Folder: {self.config.input_folder}")
            self.logger.info(f"Output Folder: {self.config.output_overlay_dir}")
            self.logger.info(f"RGB CSV Output: {self.config.output_csv_file}")
            self.logger.info(f"HSV CSV Output: {self.config.output_csv_file_hsv}")
            self.logger.info(f"LAB CSV Output: {self.config.output_csv_file_lab}")
            self.logger.info("=" * 70)
            
            # Create output directories
            self.config.output_overlay_dir.mkdir(parents=True, exist_ok=True)
            self.logger.info(f"Output directory ready: {self.config.output_overlay_dir}")
            
            # Initialize CSV outputs
            if not self.csv_manager.initialize_csv():
                self.logger.critical("CSV initialization failed - aborting pipeline")
                return False
            if not self.csv_manager_hsv.initialize_csv():
                self.logger.critical("HSV CSV initialization failed - aborting pipeline")
                return False
            if not self.csv_manager_lab.initialize_csv():  # NEW
                self.logger.critical("LAB CSV initialization failed - aborting pipeline")
                return False
            
            # Load models
            if not self.model_manager.load_models():
                self.logger.critical("Model loading failed - aborting pipeline")
                return False
            
            self.logger.info("System initialization complete!")
            return True
            
        except Exception as e:
            self.logger.critical(f"System initialization failed: {str(e)}")
            return False
    
    def get_valid_images(self) -> List[Path]:
        """Get list of valid image files for processing."""
        try:
            supported_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
            img_paths = sorted([
                p for p in self.config.input_folder.glob("*.*")
                if p.suffix.lower() in supported_extensions
            ])
            
            self.logger.info(f"Found {len(img_paths)} valid images in {self.config.input_folder}")
            
            if not img_paths:
                self.logger.warning("No valid images found!")
                for file in self.config.input_folder.glob("*.*"):
                    self.logger.debug(f"Skipped file: {file.name} (unsupported extension)")
            
            return img_paths
            
        except Exception as e:
            self.logger.error(f"Error scanning images: {str(e)}")
            return []
    
    def process_single_image(self, img_path: Path) -> bool:
        """
        Process a single image through the complete pipeline.
        
        Args:
            img_path: Path to image file
            
        Returns:
            True if processing successful, False otherwise
        """
        start_time = datetime.now()
        
        try:
            self.logger.info(f"Processing: {img_path.name}")
            
            # Load and validate image
            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None:
                raise ValueError(f"Failed to load image: {img_path}")
            
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            H, W = img_rgb.shape[:2]
            
            self.logger.debug(f"Image loaded: {W}x{H} pixels")
            
            # Step 1: Object Detection
            self.logger.debug("Step 1: Running object detection...")
            eye_results, color_results = self.image_processor.detect_objects(str(img_path))
            
            if eye_results is None or color_results is None:
                raise RuntimeError("Object detection failed")
            
            eye_detections = len(eye_results.boxes)
            color_detections = len(color_results.boxes)
            
            self.logger.info(f"  Eye detections: {eye_detections}")
            self.logger.info(f"  Color patch detections: {color_detections}")
            
            # Step 2: Eye Segmentation
            self.logger.debug("Step 2: Processing eye segmentation...")
            all_contours = []
            
            for idx, box in enumerate(eye_results.boxes):
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(W, x2), min(H, y2)
                
                crop = img_rgb[y1:y2, x1:x2]
                mask_local = self.image_processor.segment_crop(crop)
                
                contours, _ = cv2.findContours(mask_local, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    all_contours.append(cnt + np.array([[x1, y1]]))
                
                self.logger.debug(f"Eye region {idx+1}: {len(contours)} contours found")
            
            # Optional: compute region mask (not used further here)
            if all_contours:
                max_contour = max(all_contours, key=cv2.contourArea)
                region_mask = np.zeros((H, W), dtype=np.uint8)
                cv2.drawContours(region_mask, [max_contour], -1, 1, thickness=cv2.FILLED)
            
            # Step 3: Color Patch Analysis (RGB + ordering)
            self.logger.debug("Step 3: Analyzing color patches...")
            centroids = []
            for box in color_results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                centroids.append((cx, cy, box))
            
            rows = self.color_analyzer.cluster_patches_into_rows(centroids)
            
            ordered = []
            for row in rows:
                ordered.extend([b for _, _, b in sorted(row, key=lambda x: x[0])])
            
            ascending = self.color_analyzer.determine_patch_ordering(ordered, img_rgb)
            
            total = min(self.config.chart_rows * self.config.chart_cols, len(ordered))
            color_patches: List[ColorPatch] = []
            
            for idx, box in enumerate(ordered[:total]):
                uid = (idx + 1) if ascending else (total - idx)
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                
                patch = img_rgb[y1:y2, x1:x2]
                if self.config.use_median:
                    rgb_values = tuple(int(v) for v in np.median(patch, axis=(0, 1)))
                else:
                    rgb_values = tuple(int(v) for v in patch.mean(axis=(0, 1)))
                
                color_patch = ColorPatch(
                    uid=uid,
                    rgb_values=rgb_values,
                    bbox=(x1, y1, x2, y2),
                    center=((x1 + x2) // 2, (y1 + y2) // 2),
                    area=int((x2 - x1) * (y2 - y1))
                )
                color_patches.append(color_patch)
            
            self.logger.info(f"  Color patches processed: {len(color_patches)}")
            for patch in color_patches[:5]:
                self.logger.debug(f"    UID{patch.uid}: RGB{patch.rgb_values}")
            if len(color_patches) > 5:
                self.logger.debug(f"    ... and {len(color_patches) - 5} more patches")
            
            # --- HSV: compute for same bboxes & write HSV CSV ---
            img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)  # H∈[0,179], S,V∈[0,255]
            hsv_by_uid: Dict[int, Tuple[int, int, int]] = {}
            for patch in color_patches:
                x1, y1, x2, y2 = patch.bbox
                patch_hsv = img_hsv[y1:y2, x1:x2]
                if patch_hsv.size == 0:
                    hsv_by_uid[patch.uid] = (0, 0, 0)
                    continue
                if self.config.use_median:
                    h, s, v = np.median(patch_hsv, axis=(0, 1)).astype(np.int32).tolist()
                else:
                    h, s, v = patch_hsv.mean(axis=(0, 1)).astype(np.int32).tolist()
                hsv_by_uid[patch.uid] = (int(h), int(s), int(v))
            self.csv_manager_hsv.write_hsv_result(
                image_name=img_path.name,
                hsv_by_uid=hsv_by_uid,
                total_features=len(color_patches)
            )
            
            # --- LAB: compute for same bboxes & write LAB CSV ---
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)  # L∈[0,255], a/b∈[0,255] (128≈0)
            lab_by_uid: Dict[int, Tuple[int, int, int]] = {}
            for patch in color_patches:
                x1, y1, x2, y2 = patch.bbox
                patch_lab = img_lab[y1:y2, x1:x2]
                if patch_lab.size == 0:
                    lab_by_uid[patch.uid] = (0, 0, 0)
                    continue
                if self.config.use_median:
                    l, a, b = np.median(patch_lab, axis=(0, 1)).astype(np.int32).tolist()
                else:
                    l, a, b = patch_lab.mean(axis=(0, 1)).astype(np.int32).tolist()
                lab_by_uid[patch.uid] = (int(l), int(a), int(b))
            self.csv_manager_lab.write_lab_result(
                image_name=img_path.name,
                lab_by_uid=lab_by_uid,
                total_features=len(color_patches)
            )
            # --- END color space CSVs ---
            
            # Step 4: Create Visualization (if enabled)
            if self.config.save_overlays:
                self.logger.debug("Step 4: Creating visualization...")
                overlay = self.create_overlay_visualization(img_rgb, all_contours)
                overlay = self.annotate_color_patches(overlay, color_patches)
                
                out_path = self.config.output_overlay_dir / f"{Path(img_path.name).stem}_overlay.png"
                cv2.imwrite(str(out_path), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
                self.logger.debug(f"Overlay saved: {out_path}")
            
            # Calculate processing time
            processing_time = (datetime.now() - start_time).total_seconds()
            
            # Write results to RGB CSV
            self.csv_manager.write_analysis_result(
                image_name=img_path.name,
                color_patches=color_patches,
                processing_success=True
            )
            
            self.logger.info(f"✅ {img_path.name} processed successfully in {processing_time:.2f}s")
            return True
            
        except Exception as e:
            processing_time = (datetime.now() - start_time).total_seconds()
            
            # Write failed/empty rows to all CSVs
            self.csv_manager.write_analysis_result(
                image_name=img_path.name,
                color_patches=[],
                processing_success=False
            )
            self.csv_manager_hsv.write_hsv_result(
                image_name=img_path.name,
                hsv_by_uid={},
                total_features=0
            )
            self.csv_manager_lab.write_lab_result(   # NEW
                image_name=img_path.name,
                lab_by_uid={},
                total_features=0
            )
            
            self.logger.error(f"❌ Failed to process {img_path.name} in {processing_time:.2f}s: {str(e)}")
            return False
    
    def create_overlay_visualization(self, img_rgb: np.ndarray, all_contours: List) -> np.ndarray:
        """
        Create overlay image with segmentation highlighting.
        
        Args:
            img_rgb: Original RGB image
            all_contours: List of segmentation contours
            
        Returns:
            Overlay image with yellow segmentation highlight
        """
        overlay = img_rgb.copy()
        
        if all_contours:
            max_contour = max(all_contours, key=cv2.contourArea)
            H, W = img_rgb.shape[:2]
            region_mask = np.zeros((H, W), dtype=np.uint8)
            cv2.drawContours(region_mask, [max_contour], -1, 1, thickness=cv2.FILLED)
            
            # Create yellow overlay
            yellow_img = np.zeros_like(img_rgb)
            yellow_img[..., 0:2] = 255  # Yellow (255, 255, 0)
            
            overlay[region_mask == 1] = (
                overlay[region_mask == 1] * (1 - self.config.alpha) +
                yellow_img[region_mask == 1] * self.config.alpha
            ).astype(np.uint8)
            
            self.logger.debug("Applied segmentation overlay")
        
        return overlay
    
    def annotate_color_patches(self, overlay: np.ndarray, color_patches: List[ColorPatch]) -> np.ndarray:
        """
        Annotate color patches on overlay image.
        
        Args:
            overlay: Base overlay image
            color_patches: List of ColorPatch objects
            
        Returns:
            Annotated overlay image
        """
        annotated = overlay.copy()
        
        for patch in color_patches:
            x1, y1, x2, y2 = patch.bbox
            
            # Draw bounding box
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            # Add UID label
            cv2.putText(annotated, f"UID{patch.uid}", (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
            
            # Add RGB values
            cv2.putText(annotated, f"{patch.rgb_values}", (x1, y2 + 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        self.logger.debug(f"Annotated {len(color_patches)} color patches")
        return annotated
    
    def run_complete_analysis(self) -> int:
        """Run the complete analysis pipeline on all images."""
        try:
            # Initialize system
            if not self.initialize_system():
                return 0
            
            # Get valid images
            img_paths = self.get_valid_images()
            if not img_paths:
                self.logger.critical("No images to process - pipeline aborted")
                return 0
            
            # Process all images
            self.logger.info("STARTING BATCH PROCESSING")
            self.logger.info("=" * 50)
            
            successful_count = 0
            
            for img_path in tqdm(img_paths, desc="Processing images", unit="image"):
                if self.process_single_image(img_path):
                    successful_count += 1
            
            # Finalize CSV outputs
            self.csv_manager.finalize_csv()
            self.csv_manager_hsv.finalize_csv()
            self.csv_manager_lab.finalize_csv()  # NEW
            
            # Final summary
            self.logger.info("=" * 70)
            self.logger.info("PIPELINE EXECUTION COMPLETED")
            self.logger.info("=" * 70)
            self.logger.info(f"FINAL STATISTICS:")
            self.logger.info(f"  • Total images processed: {len(img_paths)}")
            self.logger.info(f"  • Successful analyses: {successful_count}")
            self.logger.info(f"  • Failed analyses: {len(img_paths) - successful_count}")
            self.logger.info(f"  • Success rate: {(successful_count/len(img_paths)*100):.1f}%")
            
            if successful_count > 0:
                self.logger.info(f"RGB results saved to: {self.config.output_csv_file}")
                self.logger.info(f"HSV results saved to: {self.config.output_csv_file_hsv}")
                self.logger.info(f"LAB results saved to: {self.config.output_csv_file_lab}")
            
            return successful_count
            
        except Exception as e:
            self.logger.critical(f"Pipeline execution failed: {str(e)}")
            return 0


def main():
    """Main execution function."""
    try:
        print("Medical Image Analysis Pipeline - RGB + HSV + LAB CSV Output")
        print("============================================================")
        
        # Initialize configuration
        config = Config()
        
        # Create and run analyzer
        analyzer = MedicalImageAnalyzer(config)
        successful_count = analyzer.run_complete_analysis()
        
        if successful_count > 0:
            print(f"\n✅ Analysis completed successfully!")
            print(f"   Successful analyses: {successful_count}")
            print(f"   RGB CSV output: {config.output_csv_file}   (77 cols: image_name + 24×RGB + total_features + avg RGB)") 
            print(f"   HSV CSV output: {config.output_csv_file_hsv} (77 cols: image_name + 24×HSV + total_features + avg HSV)")
            print(f"   LAB CSV output: {config.output_csv_file_lab} (77 cols: image_name + 24×LAB + total_features + avg LAB)")
            print(f"   Log file: {config.log_file}")
            print("\nℹ️  Color-space scales:")
            print("   • HSV (OpenCV): H∈[0,179], S∈[0,255], V∈[0,255]")
            print("   • LAB (OpenCV 8-bit): L∈[0,255], a∈[0,255] (128≈0), b∈[0,255] (128≈0)")
        else:
            print("\n❌ Analysis failed. Check the log file for details.")
            
    except KeyboardInterrupt:
        print("\n⚠️  Analysis interrupted by user")
        sys.exit(1)
    except Exception as e:
        print(f"\n💥 Critical error: {str(e)}")
        print("Check the log file for detailed error information.")
        sys.exit(1)


if __name__ == "__main__":
    main()


2026-01-20 15:26:56,505 - INFO - Medical Image Analysis Pipeline Starting
2026-01-20 15:26:56,505 - INFO - ======================================================================
2026-01-20 15:26:56,506 - INFO - Timestamp: 2026-01-20 15:26:56
2026-01-20 15:26:56,507 - INFO - Device: cpu
2026-01-20 15:26:56,507 - INFO - Input Folder: /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/Full_data
2026-01-20 15:26:56,507 - INFO - Output Folder: /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/overlay_output
2026-01-20 15:26:56,508 - INFO - RGB CSV Output: /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/RGB.csv
2026-01-20 15:26:56,508 - INFO - HSV CSV Output: /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/HSV.csv
2026-01-20 15:26:56,509 - INFO - LAB CSV Output: /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/LAB.csv
2026-01-20 15:26:56,509 - INFO - ======================================================================
2026-01-20 15:26:56,509 - INFO - Output directo

Medical Image Analysis Pipeline - RGB + HSV + LAB CSV Output


2026-01-20 15:26:57,022 - INFO - ✓ U-Net segmentation model loaded on cpu
2026-01-20 15:26:57,022 - INFO - ==================================================
2026-01-20 15:26:57,023 - INFO - ALL MODELS LOADED SUCCESSFULLY!
2026-01-20 15:26:57,023 - INFO - ==================================================
2026-01-20 15:26:57,031 - INFO - System initialization complete!
2026-01-20 15:26:57,048 - INFO - Found 2252 valid images in /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/Full_data
2026-01-20 15:26:57,049 - INFO - STARTING BATCH PROCESSING
2026-01-20 15:26:57,049 - INFO - ==================================================
Processing images:   0%|          | 0/2252 [00:00<?, ?image/s]2026-01-20 15:26:57,052 - INFO - Processing: 10.9.8.jpg
2026-01-20 15:26:57,812 - INFO -   Eye detections: 1
2026-01-20 15:26:57,813 - INFO -   Color patch detections: 24
2026-01-20 15:26:57,924 - INFO -   Color patches processed: 24
2026-01-20 15:26:58,209 - INFO - ✅ 10.9.8.jpg processed successfully


✅ Analysis completed successfully!
   Successful analyses: 2248
   RGB CSV output: /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/RGB.csv   (77 cols: image_name + 24×RGB + total_features + avg RGB)
   HSV CSV output: /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/HSV.csv (77 cols: image_name + 24×HSV + total_features + avg HSV)
   LAB CSV output: /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/LAB.csv (77 cols: image_name + 24×LAB + total_features + avg LAB)
   Log file: /home/khushi/Pixonate/New_Anemia/New/all_Sort_data/RGB/medical_analysis.log

ℹ️  Color-space scales:
   • HSV (OpenCV): H∈[0,179], S∈[0,255], V∈[0,255]
   • LAB (OpenCV 8-bit): L∈[0,255], a∈[0,255] (128≈0), b∈[0,255] (128≈0)


In [ ]:
#add_hbvalue_column_using_image_name_4
import pandas as pd
import re

df = pd.read_csv("/home/khushi/Pixonate/New_Anemia/output_images/122_RGB.csv")

def extract_hb_value(name):
    match = re.search(r'^\d+\.(\d+\.\d+)\.jpg$', str(name))
    return float(match.group(1)) if match else None

df["HBvalue"] = df["image_name"].apply(extract_hb_value)

df.to_csv(
    "/home/khushi/Pixonate/New_Anemia/output_images/122_RGB_with_HB.csv",
    index=False
)

print("✅ HB value extracted successfully!")
print(df[["image_name", "HBvalue"]].head())



✅ HB value extracted successfully!
     image_name  HBvalue
0    1.10.9.jpg     10.9
1    10.9.6.jpg      9.6
2  100.35.0.jpg     35.0
3  101.12.4.jpg     12.4
4  102.11.7.jpg     11.7


In [ ]:
#Feature_engineering_5

# This script converts raw average RGB values from image patches into a 
# rich set of mathematical color features and saves them for
# machine-learning models to predict hemoglobin (HB) levels.

import pandas as pd
import numpy as np

# === Load data ===
df = pd.read_csv(r"/home/khushi/Pixonate/New_Anemia/output_images/122_RGB_with_HB.csv")

r = df["avg_patch_r"].astype(float)
g = df["avg_patch_g"].astype(float)
b = df["avg_patch_b"].astype(float)
y = df["HBvalue"].astype(float)

eps = 1e-6
sum_rgb = r + g + b

# === Feature engineering ===
df_feat = pd.DataFrame({
    "r": r, "g": g, "b": b,
    "rgb_sum": sum_rgb,
    "rgb_mean": sum_rgb / 3.0,
    "rgb_min": np.minimum.reduce([r, g, b]),
    "rgb_max": np.maximum.reduce([r, g, b]),
    "rgb_range": np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b]),
    "rgb_std": np.std(np.vstack([r, g, b]), axis=0),

    # Normalized channels
    "r_norm": r / (sum_rgb + eps),
    "g_norm": g / (sum_rgb + eps),
    "b_norm": b / (sum_rgb + eps),

    # Ratios
    "r_g_ratio": r / (g + eps),
    "r_b_ratio": r / (b + eps),
    "g_b_ratio": g / (b + eps),

    # Differences
    "r_minus_g": r - g,
    "r_minus_b": r - b,
    "g_minus_b": g - b,

    # Products
    "r_g_prod": r * g,
    "r_b_prod": r * b,
    "g_b_prod": g * b,

    # Squares
    "r_sq": r ** 2,
    "g_sq": g ** 2,
    "b_sq": b ** 2,

    # Magnitude-like features
    "rg_mag": r * g,
    "rb_mag": r * b,
    "gb_mag": g * b,
})

# Add target back
df_feat["HBvalue"] = y


df_feat.to_excel(r"/home/khushi/Pixonate/New_Anemia/output_images/featured_rgb_values.xlsx", index=False)

print("✅ Saved:")










